In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Access it at:
# /content/drive/MyDrive/                        ← your own Drive
# /content/drive/Shareddrives/<folder-name>/     ← shared drives
# /content/drive/MyDrive/Shared with me/         ← folders shared with you

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Cell 1 — Imports & Config
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torch.nn as nn
from pathlib import Path

CHECKPOINT  = "/content/drive/MyDrive/resnet18_car_detector_best.pth"
OCCLUDE_DIR = Path("/content/drive/MyDrive/multi bird occlusion/")   # expects subdirs: 20, 40, 60, 80
BATCH_SIZE  = 32
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Occlusion levels must match your folder names
OCCLUSION_LEVELS = [20, 40, 60, 80]

In [9]:
# Cell 2 — Load Model
def load_model(checkpoint: str) -> nn.Module:
    model = models.resnet18()
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()
    return model.to(DEVICE)

model = load_model(CHECKPOINT)

In [10]:
# Cell 3 — Transforms
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [11]:
# Cell 4 — Evaluation Function
def evaluate_top1(model, loader) -> dict:
    """
    Computes Top-1 accuracy for the given loader.
    Returns correct count, total count, and accuracy.
    """
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return {"correct": correct, "total": total, "top1_acc": correct / total}

In [12]:
# Cell 5 — Run Evaluation Across Occlusion Levels
from torch.utils.data import Dataset
from PIL import Image

class OcclusionDataset(Dataset):
    def __init__(self, folder: Path, transform=None):
        self.files     = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img  # no label needed

results = {}
for level in OCCLUSION_LEVELS:
    dataset = OcclusionDataset(OCCLUDE_DIR / f"occluded_cars_dataset_{level}", transform=val_transforms)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    correct, total = 0, 0
    with torch.no_grad():
        for images in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1)
            correct += (preds == 1).sum().item()  # 1 = car
            total   += images.size(0)

    results[level] = {"correct": correct, "total": total, "top1_acc": correct / total}
    print(f"Occlusion {level:3d}% | "
          f"Top-1 Acc: {results[level]['top1_acc']:.3f} | "
          f"Correct: {correct}/{total}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Occlusion  20% | Top-1 Acc: 0.967 | Correct: 325/336
Occlusion  40% | Top-1 Acc: 0.940 | Correct: 316/336
Occlusion  60% | Top-1 Acc: 0.929 | Correct: 312/336
Occlusion  80% | Top-1 Acc: 0.955 | Correct: 321/336


In [ ]:
# Cell 6 — Plot Results
import matplotlib.pyplot as plt

levels   = list(results.keys())
top1_acc = [results[l]["top1_acc"] for l in levels]

plt.figure(figsize=(8, 5))
plt.plot(levels, top1_acc, marker="o", linewidth=2, color="steelblue")
plt.xlabel("Occlusion Level (%)")
plt.ylabel("Top-1 Accuracy")
plt.title("ResNet-18 Robustness to Occlusion")
plt.xticks(levels)
plt.ylim(0, 1)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("occlusion_robustness.png", dpi=150)
plt.show()